# Nexus · Music Studio
Opis + tekst → piosenka lub instrumental. Dwie wersje, oryginały WAV/FLAC, zapis projektu i opcjonalna normalizacja głośności.

Wybierz **Środowisko wykonawcze → Zmień typ środowiska → GPU**. Użyj świeżej sesji, bez załadowanego generatora wideo/zdjęć. Notebook pobiera duże pakiety i modele; potrzebne są RAM, dysk oraz internet. Nie gwarantuje przewagi nad Suno — do tego potrzebne są rzeczywiste odsłuchy. Koszt GPU zależy od Twojego planu.

Serwer muzyczny działa lokalnie w Colabie, a panel udostępnia chroniony hasłem tymczasowy link Gradio. Tekst i opis przetwarza to środowisko. Pobierz ZIP przed końcem sesji.

In [ ]:
import torch, shutil, sys, os, subprocess, tempfile, secrets, socket, signal
from pathlib import Path
assert torch.cuda.is_available(), "Włącz GPU w Colabie."
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "GB")
print("Wolny dysk:", round(shutil.disk_usage("/content").free / 2**30, 1), "GB")
assert shutil.which("ffmpeg"), "W tym środowisku brakuje FFmpeg potrzebnego do normalizacji."


## 1. Kod i oddzielne środowisko silnika
Instalacja silnika używa przypiętego commita ACE-Step oraz jego pliku uv.lock. Nie zastępuje PyTorcha w głównym środowisku Colaba. Może pobrać wiele GB zależności.

In [ ]:
workspace = Path(tempfile.mkdtemp(prefix="nexus-music-"))
nexus_dir = workspace / "nexus-ai"
engine_dir = workspace / "ace-step"
subprocess.run(["git", "clone", "--depth", "1", "--branch", "feat/music-studio",
                "https://github.com/Damianwojownik/nexus-ai.git", str(nexus_dir)], check=True)
engine_dir.mkdir()
subprocess.run(["git", "init"], cwd=engine_dir, check=True)
subprocess.run(["git", "remote", "add", "origin", "https://github.com/ace-step/ACE-Step-1.5.git"], cwd=engine_dir, check=True)
subprocess.run(["git", "fetch", "--depth", "1", "origin", "ca1e85fe9430179831e6bc6be790c332190a3866"], cwd=engine_dir, check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=engine_dir, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "uv", "-r", str(nexus_dir / "services/music_studio/requirements.txt")], check=True)
engine_env = os.environ.copy()
engine_env["UV_PROJECT_ENVIRONMENT"] = str(workspace / "engine-env")
subprocess.run([sys.executable, "-m", "uv", "sync", "--frozen", "--python", "3.11"],
               cwd=engine_dir, env=engine_env, check=True)
sys.path.insert(0, str(nexus_dir))
print("Kod i środowisko silnika gotowe.")


## 2. Uruchom GPU
Domyślnie model Base i LM 1.7B z offloadem. Pierwszy start pobiera modele. Przy braku pamięci wybierz LM 0.6B, krótki utwór i jedną wersję. Więcej kroków nie gwarantuje lepszego brzmienia.

Klucz serwera powstaje losowo i pozostaje w zmiennych środowiska. Nie jest wypisywany ani zapisywany do repozytorium.

In [ ]:
with socket.socket() as probe:
    assert probe.connect_ex(("127.0.0.1", 8001)) != 0, "Port 8001 jest zajęty. Zatrzymaj poprzedni serwer."
api_token = secrets.token_urlsafe(32)
engine_env.update({
    "ACESTEP_CONFIG_PATH": "acestep-v15-base",
    "ACESTEP_LM_MODEL_PATH": "acestep-5Hz-lm-1.7B",
    "ACESTEP_INIT_LLM": "true",
    "ACESTEP_LM_BACKEND": "pt",
    "ACESTEP_OFFLOAD_TO_CPU": "true",
    "ACESTEP_LM_OFFLOAD_TO_CPU": "true",
    "ACESTEP_API_KEY": api_token,
    "ACESTEP_API_HOST": "127.0.0.1",
    "ACESTEP_API_PORT": "8001",
})
log_path = workspace / "engine.log"
with log_path.open("w") as log:
    engine_process = subprocess.Popen([sys.executable, "-m", "uv", "run", "--frozen", "acestep-api"],
                                      cwd=engine_dir, env=engine_env, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
os.environ["NEXUS_MUSIC_API_URL"] = "http://127.0.0.1:8001"
os.environ["NEXUS_MUSIC_API_TOKEN"] = api_token
os.environ["NEXUS_MUSIC_OUTPUT"] = str(workspace / "projects")
os.environ["GRADIO_ANALYTICS_ENABLED"] = "False"
print("Uruchomiono proces silnika. Pierwszy start i pobieranie modeli mogą potrwać.")


## 3. Sprawdź gotowość
Uruchom tę komórkę ponownie, jeśli silnik jeszcze ładuje modele. Nie wysyła zadania muzycznego ani nie tworzy kolejnego procesu.

In [ ]:
from services.music_studio.core import AceClient
if engine_process.poll() is not None:
    print("Silnik zakończył pracę. Końcówka logu (token usunięty):")
    print(log_path.read_text(errors="replace")[-4000:].replace(api_token, "[REDACTED]"))
else:
    try:
        print("Dostępne modele:", AceClient().models())
    except Exception:
        print("Silnik jeszcze nie odpowiada. Spróbuj ponownie za chwilę.")
        print(log_path.read_text(errors="replace")[-1800:].replace(api_token, "[REDACTED]"))


## 4. Otwórz studio
Ustaw hasło minimum 12 znaków, login **nexus**. W panelu sprawdź połączenie, opisz utwór i wpisz tekst albo stwórz szkic. Na pierwszy test wybierz 30–60 sekund i jedną wersję. Zapisz project.json już podczas generowania; na końcu pobierz ZIP.

„Przestań czekać” zatrzymuje wyłącznie polling panelu. „Odbierz wynik” wraca do tego samego zadania. Normalizacja tworzy osobny plik — oceń go w porównaniu z oryginałem.

In [ ]:
from getpass import getpass
from services.music_studio.app import build_app
password = getpass("Hasło do studia (minimum 12 znaków): ")
assert len(password) >= 12, "Hasło musi mieć minimum 12 znaków."
studio = build_app()
studio.launch(share=True, auth=("nexus", password), server_name="127.0.0.1",
              max_file_size="128kb", show_error=False, debug=True)


## 5. Zakończenie pracy
Najpierw pobierz ZIP z nagraniami. Następna komórka zatrzyma Twój serwer, również jeśli nadal generuje. Project.json pozwala odebrać istniejący wynik tylko wtedy, gdy zadanie i pliki pozostają dostępne na serwerze.

In [ ]:
studio.close()
if engine_process.poll() is None:
    os.killpg(engine_process.pid, signal.SIGTERM)
try:
    engine_process.wait(timeout=15)
except subprocess.TimeoutExpired:
    print("Proces nadal się zamyka. Możesz zakończyć środowisko Colab po pobraniu plików.")
